# Genotype Data Preprocessing

## Description

This mini-protocol walks a new user through preparing genotype data so it is ready for QTL analysis. Starting from per-gene/region VCF files, it cleans and standardizes variants, exports to PLINK format, applies genotype-level quality control, and splits the QC'd data by chromosome.

It chains the following steps together:
1. **VCF quality control** (`VCF_QC.ipynb`): variant preprocessing (split multiallelics, left-normalize indels, annotate dbSNP), variant-level QC (DP/GQ/AB filters; drop monomorphic, high-missingness, and HWE-failing variants), and summary statistics.
2. **VCF to PLINK** (`genotype_formatting.ipynb`): convert the QC'd VCFs to PLINK1 format and merge per-chromosome files into one dataset.
3. **PLINK quality control** (`GWAS_QC.ipynb`): filter variants/samples by MAF, missingness (geno/mind), and Hardy-Weinberg equilibrium.
4. **Split by chromosome** (`genotype_formatting.ipynb`): produce one PLINK file set per chromosome for downstream cis analysis.

> Note: principal-component analysis (PCA) and the genetic relationship matrix (GRM) are covered in the next mini-protocol, `3_genotype_pca.ipynb`, not here. The toy inputs use 60 samples named `SAMPLE_001 ... SAMPLE_060`, consistent with the phenotype MWE.

## Input Files

| File | Description |
|------|-------------|
| `input/genotype/protocol_example.genotype.ENSG00000073921.vcf.gz` | Example per-gene VCF (with `chr` prefix), 60 samples, used for the VCF QC demonstration. |
| `input/genotype/protocol_example.genotype.chr22.vcf.gz` | Example chromosome-level WGS VCF, 60 samples, used for the VCF-to-PLINK conversion. |
| `input/reference_data/00-All.add_chr.variants.gz` | dbSNP variants (chrom, start, end, rsID per SNP) used to annotate known variants with rsIDs. |
| `input/reference_data/GRCh38_full_analysis_set_plus_decoy_hla.noALT_noHLA_noDecoy_ERCC.fasta` | Reference genome FASTA used to left-normalize indels and correct REF/ALT. |

## Steps

### **Step 1.** VCF Quality Control

First, inspect the raw genotype VCF and the dbSNP annotation file before running QC.

In [ ]:
setwd('/home/ubuntu/xqtl_protocol_exercise')
library(data.table)
# genotype VCF before QC
geno = fread('data/WGS/vcf/ENSG00000073921.variants.add_chr.vcf.gz')
dim(geno)
geno[1:4,1:11]

```
#CHROM  POS     ID      REF     ALT     QUAL    FILTER  INFO    FORMAT  SAMPLE_001      SAMPLE_002      SAMPLE_003      SAMPLE_004
chr11   84957209        chr11:84957209_G_C      G       C       .       .       PR;AC=32;AN=98  GT      0/0     0/0     0/1     0/1
chr11   84957210        chr11:84957210_C_T      C       T       .       .       PR;AC=0;AN=98   GT      0/0     0/0     0/0     0/0
chr11   84957254        chr11:84957254_A_C      A       C       .       .       PR;AC=0;AN=98   GT      0/0     0/0     0/0     0/0
chr11   84957263        chr11:84957263_C_T      C       T       .       .       PR;AC=0;AN=98   GT      0/0     0/0     0/0     0/0
chr11   84957273        chr11:84957273_G_A      G       A       .       .       PR;AC=0;AN=98   GT      0/0     0/0     0/0     0/0
chr11   84957281        chr11:84957281_T_C      T       C       .       .       PR;AC=0;AN=98   GT      0/0     0/0     0/0     0/0
chr11   84957283        chr11:84957283_C_T      C       T       .       .       PR;AC=0;AN=98   GT      0/0     0/0     0/0     0/0
chr11   84957288        chr11:84957288_T_C      T       C       .       .       PR;AC=0;AN=98   GT      0/0     0/0     0/0     0/0
chr11   84957364        chr11:84957364_T_C      T       C       .       .       PR;AC=0;AN=98   GT      0/0     0/0     0/0     0/0
```

In [ ]:
# dbsnp-variants file to annotate rsid 
# chrom start end rsid for each snp
cd /home/ubuntu/xqtl_protocol_exercise
zcat reference_data/00-All.add_chr.variants.gz | head

```
chr11   60994   60994   rs1452162522
chr11   61118   61118   rs867959887
chr11   61119   61119   rs1187622598
chr11   61155   61155   rs899119276
chr11   61194   61194   rs1245081080
chr11   61248   61248   rs367559610
chr11   61251   61251   rs370183475
chr11   61284   61284   rs1223817784
chr11   61350   61350   rs1421732777
chr11   61367   61367   rs374502399
```

This step cleans the raw VCF and filters low-quality variants and genotypes. Internally it performs three things:

1. **Variant preprocessing** - split multi-allelic records into bi-allelic, left-normalize indels and correct REF/ALT against the reference FASTA, and annotate known variants with rsIDs from dbSNP. After this the `ID` field changes from `.`/`chr:pos` to `rsXXXX` where matched, and each record is bi-allelic.

2. **Variant-level QC** - for each genotype, filter by depth (DP), genotype quality (GQ), and allele balance (AB); then drop monomorphic sites (no heterozygosity across samples), variants with high missingness, and variants failing the optional HWE threshold. Low-confidence genotypes are set to `./.`, variants with no remaining informative genotypes are removed, and the file becomes cleaner and smaller.

3. **Summary statistics** - using `bcftools stats` and `SnpSift tstv`, compute total variants, SNP/indel counts, missingness, heterozygosity, and the transition/transversion (TS/TV) ratio, separated into known (with rsID) and novel (no rsID) variants.

The QC'd data is also exported to PLINK format for the next step.

In [ ]:
sos run pipeline/VCF_QC.ipynb qc \
    --genoFile data/WGS/vcf/ENSG00000073921.variants.add_chr.vcf.gz \
    --dbsnp-variants reference_data/00-All.add_chr.variants.gz \
    --reference-genome reference_data/GRCh38_full_analysis_set_plus_decoy_hla.noALT_noHLA_noDecoy_ERCC.fasta \
    --gt-only-vcf-qc \
    --cwd output/vcf/ 

### **Step 2.** Convert VCF to PLINK

This step converts the (QC'd) VCF files to PLINK1 format, then merges the per-chromosome PLINK files into a single dataset. PLINK1 (traditional) format consists of three files: `.bed` (binary genotype data), `.bim` (variant information: chromosome, position, variant ID, etc.), and `.fam` (sample information: family ID, individual ID, etc.).

In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
sos run pipeline/genotype_formatting.ipynb vcf_to_plink \
    --genoFile `ls data/WGS/vcf/wgs.chr*.random.vcf.gz` \
    --cwd output/plink/ 

### **Step 3.** PLINK Quality Control

This step applies genotype-level QC to the merged PLINK data with the `qc_no_prune` workflow (basic QC filters, no LD pruning). Its goal is to filter SNPs and select individuals based on quality-control criteria so the genotype data is high quality and free of potential errors or biases before analysis.

Key parameters:
- `--genoFile`: the input PLINK genotype data.
- `--maf-filter` / `--maf-max-filter`: minimum / maximum Minor Allele Frequency (MAF) thresholds.
- `--mac-filter` / `--mac-max-filter`: minimum / maximum Minor Allele Count (MAC) thresholds.
- `--geno-filter`: maximum missingness per variant.
- `--mind-filter`: maximum missingness per sample.
- `--hwe-filter`: Hardy-Weinberg Equilibrium (HWE) filter threshold.
- `--rm-dups`: remove duplicate variants.
- `--meta-only`: output only SNP and sample lists rather than full genotype files.

The output is a file (set) with the suffix `.plink_qc` (e.g. `.bed`); the exact format depends on `--meta-only`.

In [ ]:
# merge plink bed into 1
sos run pipeline/genotype_formatting.ipynb merge_plink \
    --genoFile `ls output/plink/wgs.chr*.random.bed` \
    --name wgs.merged \
    --cwd output/plink/ 

### **Step 4.** Split QC'd PLINK by Chromosome

This step splits the QC'd PLINK data into one file set per chromosome, which is the format downstream cis analysis expects.

In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
sos run pipeline/GWAS_QC.ipynb qc_no_prune \
   --cwd output/plink \
   --genoFile output/plink/wgs.merged.bed \
   --geno-filter 0.1 \
   --mind-filter 0.1 \
   --hwe-filter 1e-08 \
   --mac-filter 0 

## Output Files

| File | Description |
|------|-------------|
| `output/vcf/*.leftnorm.bcftools_qc.vcf.gz` | QC'd, normalized, dbSNP-annotated VCF (output of step i). |
| `output/vcf/*.known_variant_sumstats`, `*.novel_variant_sumstats`, `*.snipsift_tstv` | VCF summary statistics, split into known/novel variants (output of step i). |
| `output/plink/protocol_example.genotype.merged.{bed,bim,fam}` | Merged PLINK dataset (output of step ii). |
| `output/plink/protocol_example.genotype.merged.plink_qc.{bed,bim,fam}` | QC'd PLINK dataset (output of step iii). |
| `output/genotype_by_chrom/*.chrN.{bed,bim,fam}` | Per-chromosome PLINK file sets (output of step iv). |

## Anticipated Results

Genotype preprocessing produces a cleaned, QC'd, per-chromosome PLINK genotype dataset (60 samples, `SAMPLE_001 ... SAMPLE_060`) that is ready for genotype PCA (`3_genotype_pca.ipynb`) and downstream QTL association analysis.

In [ ]:
sos run pipeline/genotype_formatting.ipynb genotype_by_chrom \
    --genoFile output/plink/wgs.merged.plink_qc.bed \
    --cwd output/genotype_by_chrom \
    --chrom `cut -f 1 output/plink/wgs.merged.plink_qc.bim | uniq | sed "s/chr//g"`

/home/al4225/.pixi/envs/python/lib/python3.12/site-packages/sos/targets.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
INFO: Running genotype_by_chrom_1: 
INFO: genotype_by_chrom_1 (index=3) is completed.
INFO: genotype_by_chrom_1 (index=6) is completed.
INFO: genotype_by_chrom_1 (index=0) is completed.
INFO: genotype_by_chrom_1 (index=5) is completed.
INFO: genotype_by_chrom_1 (index=1) is completed.
INFO: genotype_by_chrom_1 (index=4) is completed.
INFO: genotype_by_chrom_1 (index=2) is completed.
INFO: genotype_by_chrom_1 (index=9) is completed.
INFO: genotype_by_chrom_1 (index=7) is completed.
INFO: genotype_by_chrom_1 (index=12) is completed.
INFO: genotype_by_chrom_1 (index=11) is completed.
INFO: genotype_by_chrom_1 (index=13) is completed.
INFO: genotype_by